# MB52 — Estoque por Depósito

**Tabela:** `dev_procurement.corp_curated.tbl_ds_log_mb52`
**Transação SAP:** MB52 · **Colunas:** 23
**Clustering declarado:** `cod_material`, `cod_centro`

---

## Objetivo
Mapear o comportamento desta tabela **antes** de qualquer comparação com o SAP.
O resultado alimenta a base de conhecimento do agente de validação e define o cenário de teste.

## Como usar
1. Execute a célula **1** para criar os widgets, depois ajuste os filtros no topo (opcional).
2. Execute a célula **2** — ela cria a view `base`, usada por todas as demais.
3. Execute as células na ordem e leia a coluna **`veredito`** de cada resultado.
4. Exporte o notebook executado para a pasta de conhecimento do agente.

## Aviso metodológico
Contagem de linhas **não** é evidência de qualidade. Erros de colapso de granularidade
preservam o total. Ver seções **4**, **5** e **14**.


## 1. Widgets de recorte

Execute uma vez. Deixe vazio para analisar a base completa.

In [0]:
-- 1. WIDGETS (deixe vazio = sem filtro)
-- Parametros criados na barra de widgets do notebook.
SELECT
  :f_cod_centro AS f_cod_centro,
  :f_tp_material AS f_tp_material,
  :f_cod_deposito AS f_cod_deposito;

## 2. View `base`

Aplica os filtros dos widgets uma única vez. **Todas** as células seguintes consultam `base`.

In [0]:
-- 2. VIEW BASE (aplica os filtros dos widgets)
CREATE OR REPLACE TEMP VIEW base AS
SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mb52
WHERE (:f_cod_centro = '' OR `cod_centro` = :f_cod_centro)
  AND (:f_tp_material = '' OR `tp_material` = :f_tp_material)
  AND (:f_cod_deposito = '' OR `cod_deposito` = :f_cod_deposito);

SELECT COUNT(*) AS linhas_na_base FROM base;

## 3. Metadados e histórico de carga

Formato, particionamento, clustering real e última atualização.
Divergência entre clustering declarado e chave real é o primeiro indício de problema.

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_log_mb52;

In [0]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_log_mb52;

In [0]:
-- Ultimas operacoes de escrita (falha se for view ou nao-Delta)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_log_mb52 LIMIT 20;

## 4. Granularidade real

`linhas ÷ chaves distintas`. Razão maior que 1,00 significa que existe uma dimensão
adicional multiplicando as linhas — é preciso descobrir **qual** (seção 14).

In [0]:
-- 4. GRANULARIDADE REAL: linhas / chaves distintas
-- Razao > 1,00 significa que existe dimensao adicional multiplicando linhas.
WITH t AS (SELECT COUNT(*) AS total FROM base),
g AS (
SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM base)
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito` FROM base)
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial` FROM base)
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial + dateingest' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, `dateingest` FROM base)
)
SELECT g.chave, t.total AS linhas, g.combinacoes_distintas,
       ROUND(t.total / g.combinacoes_distintas, 4) AS linhas_por_chave,
       CASE WHEN g.combinacoes_distintas = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade por chave candidata

Quantas combinações se repetem e qual o pior caso.

In [0]:
-- 5. DUPLICIDADE POR CHAVE
SELECT 'cod_material + cod_centro' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd FROM base GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, `cod_centro`, `cod_deposito`, COUNT(*) AS qtd FROM base GROUP BY `cod_material`, `cod_centro`, `cod_deposito` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, COUNT(*) AS qtd FROM base GROUP BY `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito + tp_estoque_especial + dateingest' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, `dateingest`, COUNT(*) AS qtd FROM base GROUP BY `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, `dateingest` HAVING COUNT(*) > 1)
ORDER BY chaves_repetidas DESC;

## 6. Varredura de preenchimento — TODAS as colunas

**Seção mais importante do notebook.**

Detecta coluna nunca carregada. Em validação anterior, esta análise revelou 7 colunas
100% nulas no Datalake — uma delas preenchida em **97,9%** dos registros do SAP.
Este erro **não aparece** em teste por amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
-- Detecta coluna nunca carregada. Secao mais importante do notebook.
WITH t AS (SELECT COUNT(*) AS total FROM base),
perf AS (
  SELECT stack(23,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'tp_material', 'string', COUNT_IF(`tp_material` IS NULL), COUNT_IF(`tp_material` IS NOT NULL AND lower(trim(`tp_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_material`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_mercadorias', 'string', COUNT_IF(`tp_grupo_mercadorias` IS NULL), COUNT_IF(`tp_grupo_mercadorias` IS NOT NULL AND lower(trim(`tp_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$'),
    'nm_centro', 'string', COUNT_IF(`nm_centro` IS NULL), COUNT_IF(`nm_centro` IS NOT NULL AND lower(trim(`nm_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_centro`) RLIKE '^0+([.,]0+)?$'),
    'ind_eliminacao_deposito', 'string', COUNT_IF(`ind_eliminacao_deposito` IS NULL), COUNT_IF(`ind_eliminacao_deposito` IS NOT NULL AND lower(trim(`ind_eliminacao_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_eliminacao_deposito`) RLIKE '^0+([.,]0+)?$'),
    'tp_estoque_especial', 'string', COUNT_IF(`tp_estoque_especial` IS NULL), COUNT_IF(`tp_estoque_especial` IS NOT NULL AND lower(trim(`tp_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'qt_utilizacao_livre', 'decimal(13,3)', COUNT_IF(`qt_utilizacao_livre` IS NULL), 0L, COUNT_IF(`qt_utilizacao_livre` = 0),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'vl_utilizacao_livre', 'double', COUNT_IF(`vl_utilizacao_livre` IS NULL), 0L, COUNT_IF(`vl_utilizacao_livre` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'qt_transito', 'decimal(13,3)', COUNT_IF(`qt_transito` IS NULL), 0L, COUNT_IF(`qt_transito` = 0),
    'vl_transito', 'double', COUNT_IF(`vl_transito` IS NULL), 0L, COUNT_IF(`vl_transito` = 0),
    'qt_controle_qualidade', 'decimal(13,3)', COUNT_IF(`qt_controle_qualidade` IS NULL), 0L, COUNT_IF(`qt_controle_qualidade` = 0),
    'vl_controle_qualidade', 'double', COUNT_IF(`vl_controle_qualidade` IS NULL), 0L, COUNT_IF(`vl_controle_qualidade` = 0),
    'qt_estoque_bloqueado', 'decimal(13,3)', COUNT_IF(`qt_estoque_bloqueado` IS NULL), 0L, COUNT_IF(`qt_estoque_bloqueado` = 0),
    'vl_estoque_bloqueado', 'double', COUNT_IF(`vl_estoque_bloqueado` IS NULL), 0L, COUNT_IF(`vl_estoque_bloqueado` = 0),
    'cod_estoque_especial', 'string', COUNT_IF(`cod_estoque_especial` IS NULL), COUNT_IF(`cod_estoque_especial` IS NOT NULL AND lower(trim(`cod_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM base
)
SELECT
  p.coluna,
  p.tipo,
  p.nulos,
  p.vazios,
  p.zeros,
  t.total - p.nulos - p.vazios - p.zeros                              AS uteis,
  ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
  CASE
    WHEN p.nulos = t.total                                      THEN '1. 100% NULO'
    WHEN t.total - p.nulos - p.vazios - p.zeros <= 0            THEN '2. SEM VALOR UTIL'
    WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO (<1%)'
    ELSE '9. ok'
  END AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Valores distintos por coluna. Coluna constante é candidata a default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM base),
card AS (
  SELECT stack(23,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'tp_material', 'string', approx_count_distinct(`tp_material`),
    'tp_grupo_mercadorias', 'string', approx_count_distinct(`tp_grupo_mercadorias`),
    'nm_centro', 'string', approx_count_distinct(`nm_centro`),
    'ind_eliminacao_deposito', 'string', approx_count_distinct(`ind_eliminacao_deposito`),
    'tp_estoque_especial', 'string', approx_count_distinct(`tp_estoque_especial`),
    'qt_utilizacao_livre', 'decimal(13,3)', approx_count_distinct(`qt_utilizacao_livre`),
    'sg_unidade_medida_basica', 'string', approx_count_distinct(`sg_unidade_medida_basica`),
    'vl_utilizacao_livre', 'double', approx_count_distinct(`vl_utilizacao_livre`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'qt_transito', 'decimal(13,3)', approx_count_distinct(`qt_transito`),
    'vl_transito', 'double', approx_count_distinct(`vl_transito`),
    'qt_controle_qualidade', 'decimal(13,3)', approx_count_distinct(`qt_controle_qualidade`),
    'vl_controle_qualidade', 'double', approx_count_distinct(`vl_controle_qualidade`),
    'qt_estoque_bloqueado', 'decimal(13,3)', approx_count_distinct(`qt_estoque_bloqueado`),
    'vl_estoque_bloqueado', 'double', approx_count_distinct(`vl_estoque_bloqueado`),
    'cod_estoque_especial', 'string', approx_count_distinct(`cod_estoque_especial`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM base
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE
         WHEN c.distintos <= 1                   THEN '1. CONSTANTE (1 valor)'
         WHEN c.distintos <= 3                   THEN '2. cardinalidade muito baixa'
         WHEN c.distintos > t.total * 0.95       THEN '3. candidata a identificador'
         ELSE '9. normal'
       END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada coluna. Um valor concentrando mais de 99% da base
indica possível default de carga em vez de dado real.

In [0]:
-- 8. DOMINIO DAS COLUNAS CATEGORICAS (top 8 de cada)
-- Valor concentrando >99% indica possivel default de carga.
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_material' AS coluna, CAST(`tp_material` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_mercadorias' AS coluna, CAST(`tp_grupo_mercadorias` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_grupo_mercadorias` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_estoque_especial' AS coluna, CAST(`tp_estoque_especial` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_estoque_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_basica' AS coluna, CAST(`sg_unidade_medida_basica` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `sg_unidade_medida_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_eliminacao_deposito' AS coluna, CAST(`ind_eliminacao_deposito` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_eliminacao_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_estoque_especial' AS coluna, CAST(`cod_estoque_especial` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_estoque_especial` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

**Atenção ao tipo:** quantidade costuma usar `decimal`, mas valor monetário
frequentemente usa `double` — risco de arredondamento na conciliação financeira.

In [0]:
-- 9. PERFIL DOS CAMPOS NUMERICOS
-- Tipo DOUBLE em valor monetario = risco de arredondamento na conciliacao.
SELECT * FROM (
  SELECT stack(8,
    'qt_utilizacao_livre', 'decimal(13,3)', COUNT(`qt_utilizacao_livre`), CAST(MIN(`qt_utilizacao_livre`) AS DOUBLE), CAST(MAX(`qt_utilizacao_livre`) AS DOUBLE), CAST(AVG(`qt_utilizacao_livre`) AS DOUBLE), CAST(percentile_approx(`qt_utilizacao_livre`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_utilizacao_livre`, 0.95) AS DOUBLE), COUNT_IF(`qt_utilizacao_livre` < 0), COUNT_IF(`qt_utilizacao_livre` = 0),
    'vl_utilizacao_livre', 'double', COUNT(`vl_utilizacao_livre`), CAST(MIN(`vl_utilizacao_livre`) AS DOUBLE), CAST(MAX(`vl_utilizacao_livre`) AS DOUBLE), CAST(AVG(`vl_utilizacao_livre`) AS DOUBLE), CAST(percentile_approx(`vl_utilizacao_livre`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_utilizacao_livre`, 0.95) AS DOUBLE), COUNT_IF(`vl_utilizacao_livre` < 0), COUNT_IF(`vl_utilizacao_livre` = 0),
    'qt_transito', 'decimal(13,3)', COUNT(`qt_transito`), CAST(MIN(`qt_transito`) AS DOUBLE), CAST(MAX(`qt_transito`) AS DOUBLE), CAST(AVG(`qt_transito`) AS DOUBLE), CAST(percentile_approx(`qt_transito`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_transito`, 0.95) AS DOUBLE), COUNT_IF(`qt_transito` < 0), COUNT_IF(`qt_transito` = 0),
    'vl_transito', 'double', COUNT(`vl_transito`), CAST(MIN(`vl_transito`) AS DOUBLE), CAST(MAX(`vl_transito`) AS DOUBLE), CAST(AVG(`vl_transito`) AS DOUBLE), CAST(percentile_approx(`vl_transito`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_transito`, 0.95) AS DOUBLE), COUNT_IF(`vl_transito` < 0), COUNT_IF(`vl_transito` = 0),
    'qt_controle_qualidade', 'decimal(13,3)', COUNT(`qt_controle_qualidade`), CAST(MIN(`qt_controle_qualidade`) AS DOUBLE), CAST(MAX(`qt_controle_qualidade`) AS DOUBLE), CAST(AVG(`qt_controle_qualidade`) AS DOUBLE), CAST(percentile_approx(`qt_controle_qualidade`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_controle_qualidade`, 0.95) AS DOUBLE), COUNT_IF(`qt_controle_qualidade` < 0), COUNT_IF(`qt_controle_qualidade` = 0),
    'vl_controle_qualidade', 'double', COUNT(`vl_controle_qualidade`), CAST(MIN(`vl_controle_qualidade`) AS DOUBLE), CAST(MAX(`vl_controle_qualidade`) AS DOUBLE), CAST(AVG(`vl_controle_qualidade`) AS DOUBLE), CAST(percentile_approx(`vl_controle_qualidade`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_controle_qualidade`, 0.95) AS DOUBLE), COUNT_IF(`vl_controle_qualidade` < 0), COUNT_IF(`vl_controle_qualidade` = 0),
    'qt_estoque_bloqueado', 'decimal(13,3)', COUNT(`qt_estoque_bloqueado`), CAST(MIN(`qt_estoque_bloqueado`) AS DOUBLE), CAST(MAX(`qt_estoque_bloqueado`) AS DOUBLE), CAST(AVG(`qt_estoque_bloqueado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_bloqueado`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_estoque_bloqueado`, 0.95) AS DOUBLE), COUNT_IF(`qt_estoque_bloqueado` < 0), COUNT_IF(`qt_estoque_bloqueado` = 0),
    'vl_estoque_bloqueado', 'double', COUNT(`vl_estoque_bloqueado`), CAST(MIN(`vl_estoque_bloqueado`) AS DOUBLE), CAST(MAX(`vl_estoque_bloqueado`) AS DOUBLE), CAST(AVG(`vl_estoque_bloqueado`) AS DOUBLE), CAST(percentile_approx(`vl_estoque_bloqueado`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_estoque_bloqueado`, 0.95) AS DOUBLE), COUNT_IF(`vl_estoque_bloqueado` < 0), COUNT_IF(`vl_estoque_bloqueado` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM base
)
ORDER BY coluna;

## 10.1 Datas em tipo nativo

In [0]:
-- 10.1 DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(1,
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date()), COUNT_IF(`dateingest` < DATE'1990-01-01')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, anteriores_1990)
  FROM base
)
ORDER BY coluna;

## 11. Códigos — zeros à esquerda, espaços e formato

**Armadilha conhecida:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá **0% de match**.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- 11. CODIGOS: ZEROS A ESQUERDA, ESPACOS E FORMATO
-- ARMADILHA: SAP grava '425263', Datalake grava '000000000000425263'.
-- Sem normalizar, o join da 0% de match.
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes_ao_remover_zeros,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN com_espacos > 0 THEN 'tem espacos' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END,
         CASE WHEN tipo LIKE '%int%' OR tipo LIKE 'big%'
              THEN 'TIPO NUMERICO - zeros a esquerda JA perdidos' END
       ) AS alertas
FROM (
  SELECT stack(3,
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_material` AS STRING) <> trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro` AS STRING) <> trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_deposito', 'string', COUNT_IF(CAST(`cod_deposito` AS STRING) IS NULL OR trim(CAST(`cod_deposito` AS STRING)) = ''), MIN(length(trim(CAST(`cod_deposito` AS STRING)))), MAX(length(trim(CAST(`cod_deposito` AS STRING)))), COUNT_IF(trim(CAST(`cod_deposito` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_deposito` AS STRING) <> trim(CAST(`cod_deposito` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_deposito` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_deposito` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
        distintos_bruto, distintos_sem_zeros)
  FROM base
)
ORDER BY coluna;

## 12. Amostra de linhas completas

O dado como ele realmente está: formato de código, decimais, datas e nulos.

In [0]:
-- 12. AMOSTRA
SELECT * FROM base LIMIT 20;

In [0]:
-- 12.1 AMOSTRA ALEATORIA
SELECT * FROM base ORDER BY rand() LIMIT 10;

## 13. Distribuição por dimensão de recorte

Base para escolher o cenário de teste: volume viável (10 mil a 300 mil linhas)
contendo os casos-limite identificados nas seções anteriores.

In [0]:
-- 13. DISTRIBUICAO POR cod_centro
SELECT `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_centro`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR tp_material
SELECT `tp_material`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `tp_material`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_deposito
SELECT `cod_deposito`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_deposito`
ORDER BY linhas DESC
LIMIT 40;

## 14. Duplicidade — o que diferencia as linhas repetidas?

Analisando pela chave **cod_material + cod_centro**.

**Regra crítica:** linhas **idênticas** = duplicata real (erro de carga).
Linhas **distintas** = granularidade adicional legítima (split valuation, lote, tipo de avaliação).

São problemas diferentes com tratamentos diferentes. Em validação anterior, 5 linhas do mesmo
material eram todas distintas, diferenciadas por um campo que sequer existia nos extratos do SAP.

In [0]:
-- 14. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd
FROM base
GROUP BY `cod_material`, `cod_centro`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
-- 14.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS?
-- REGRA: linhas identicas = duplicata real (erro de carga).
--        linhas distintas = granularidade adicional legitima (split valuation, lote...).
WITH dup AS (
  SELECT `cod_material`, `cod_centro` FROM base GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1
),
d AS (
  SELECT b.* FROM base b JOIN dup USING (`cod_material`, `cod_centro`)
),
agg AS (
  SELECT `cod_material`, `cod_centro`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `tp_material`) AS `tp_material`,
         COUNT(DISTINCT `tp_grupo_mercadorias`) AS `tp_grupo_mercadorias`,
         COUNT(DISTINCT `nm_centro`) AS `nm_centro`,
         COUNT(DISTINCT `ind_eliminacao_deposito`) AS `ind_eliminacao_deposito`,
         COUNT(DISTINCT `tp_estoque_especial`) AS `tp_estoque_especial`,
         COUNT(DISTINCT `qt_utilizacao_livre`) AS `qt_utilizacao_livre`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `vl_utilizacao_livre`) AS `vl_utilizacao_livre`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `qt_transito`) AS `qt_transito`,
         COUNT(DISTINCT `vl_transito`) AS `vl_transito`,
         COUNT(DISTINCT `qt_controle_qualidade`) AS `qt_controle_qualidade`,
         COUNT(DISTINCT `vl_controle_qualidade`) AS `vl_controle_qualidade`,
         COUNT(DISTINCT `qt_estoque_bloqueado`) AS `qt_estoque_bloqueado`,
         COUNT(DISTINCT `vl_estoque_bloqueado`) AS `vl_estoque_bloqueado`,
         COUNT(DISTINCT `cod_estoque_especial`) AS `cod_estoque_especial`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `cod_material`, `cod_centro`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1
            THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(21,
    'desc_material', MAX(`desc_material`),
    'cod_deposito', MAX(`cod_deposito`),
    'tp_material', MAX(`tp_material`),
    'tp_grupo_mercadorias', MAX(`tp_grupo_mercadorias`),
    'nm_centro', MAX(`nm_centro`),
    'ind_eliminacao_deposito', MAX(`ind_eliminacao_deposito`),
    'tp_estoque_especial', MAX(`tp_estoque_especial`),
    'qt_utilizacao_livre', MAX(`qt_utilizacao_livre`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'vl_utilizacao_livre', MAX(`vl_utilizacao_livre`),
    'cod_moeda', MAX(`cod_moeda`),
    'qt_transito', MAX(`qt_transito`),
    'vl_transito', MAX(`vl_transito`),
    'qt_controle_qualidade', MAX(`qt_controle_qualidade`),
    'vl_controle_qualidade', MAX(`vl_controle_qualidade`),
    'qt_estoque_bloqueado', MAX(`qt_estoque_bloqueado`),
    'vl_estoque_bloqueado', MAX(`vl_estoque_bloqueado`),
    'cod_estoque_especial', MAX(`cod_estoque_especial`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 15. Freshness — atualidade da carga

In [0]:
-- 15. FRESHNESS
SELECT `dateingest` AS data_carga,
       COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS chaves_distintas
FROM base
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 15.1 SNAPSHOT OU HISTORICO?
SELECT MIN(`dateingest`) AS primeira_carga,
       MAX(`dateingest`) AS ultima_carga,
       COUNT(DISTINCT `dateingest`) AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT `dateingest`) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir a data de carga'
       END AS veredito
FROM base;

## 16. Análises específicas — MB52

### 16.1 Colapso de depósito
No SAP a MB52 tem granularidade **material + centro + depósito + tipo de estoque especial**.
O clustering do Datalake declara apenas **material + centro**.

Se um material tem N depósitos no SAP e 1 linha no Datalake, há colapso —
e a perda **não aparece** na contagem total.

In [0]:
-- 16.1 QUANTOS DEPOSITOS POR MATERIAL+CENTRO
WITH g AS (
  SELECT cod_material, cod_centro,
         COUNT(DISTINCT cod_deposito) AS qt_depositos,
         COUNT(*) AS linhas
  FROM base GROUP BY cod_material, cod_centro
)
SELECT qt_depositos,
       COUNT(*) AS materiais,
       SUM(linhas - 1) AS linhas_excedentes
FROM g GROUP BY qt_depositos ORDER BY qt_depositos;

In [0]:
-- 16.1b MATERIAIS COM MAIS DEPOSITOS
SELECT cod_material, cod_centro,
       COUNT(DISTINCT cod_deposito) AS qt_depositos,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_deposito))) AS depositos
FROM base
GROUP BY cod_material, cod_centro
HAVING COUNT(DISTINCT cod_deposito) > 1
ORDER BY qt_depositos DESC
LIMIT 25;

### 16.2 Coerência entre quantidade e valor
Se a quantidade é zero, o valor deveria ser zero — e vice-versa.

In [0]:
-- 16.2 COERENCIA QUANTIDADE x VALOR
SELECT 'qt_utilizacao_livre / vl_utilizacao_livre' AS par,
       COUNT_IF(`qt_utilizacao_livre` <> 0 OR `vl_utilizacao_livre` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_utilizacao_livre` = 0 AND `vl_utilizacao_livre` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_utilizacao_livre` <> 0 AND `vl_utilizacao_livre` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_utilizacao_livre` < 0) AS qtd_negativa,
       COUNT_IF(`vl_utilizacao_livre` < 0) AS valor_negativo
  FROM base
UNION ALL
SELECT 'qt_transito / vl_transito' AS par,
       COUNT_IF(`qt_transito` <> 0 OR `vl_transito` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_transito` = 0 AND `vl_transito` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_transito` <> 0 AND `vl_transito` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_transito` < 0) AS qtd_negativa,
       COUNT_IF(`vl_transito` < 0) AS valor_negativo
  FROM base
UNION ALL
SELECT 'qt_controle_qualidade / vl_controle_qualidade' AS par,
       COUNT_IF(`qt_controle_qualidade` <> 0 OR `vl_controle_qualidade` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_controle_qualidade` = 0 AND `vl_controle_qualidade` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_controle_qualidade` <> 0 AND `vl_controle_qualidade` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_controle_qualidade` < 0) AS qtd_negativa,
       COUNT_IF(`vl_controle_qualidade` < 0) AS valor_negativo
  FROM base
UNION ALL
SELECT 'qt_estoque_bloqueado / vl_estoque_bloqueado' AS par,
       COUNT_IF(`qt_estoque_bloqueado` <> 0 OR `vl_estoque_bloqueado` <> 0) AS linhas_com_estoque,
       COUNT_IF(`qt_estoque_bloqueado` = 0 AND `vl_estoque_bloqueado` <> 0) AS qtd_zero_valor_nao,
       COUNT_IF(`qt_estoque_bloqueado` <> 0 AND `vl_estoque_bloqueado` = 0) AS qtd_nao_valor_zero,
       COUNT_IF(`qt_estoque_bloqueado` < 0) AS qtd_negativa,
       COUNT_IF(`vl_estoque_bloqueado` < 0) AS valor_negativo
  FROM base;

## 90. Integridade referencial cruzada _(opcional)_

Confere se os códigos desta tabela existem nas tabelas de referência.
Execute apenas se as outras tabelas estiverem acessíveis no mesmo ambiente.

In [0]:
-- 90. INTEGRIDADE: cod_material -> dev_procurement.corp_curated.tbl_ds_mdm_mm60.cod_material
WITH loc AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM base WHERE `cod_material` IS NOT NULL AND trim(CAST(`cod_material` AS STRING)) <> ''
),
ref AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
)
SELECT (SELECT COUNT(*) FROM loc) AS codigos_distintos_aqui,
       (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k)) AS sem_correspondencia,
       ROUND(100.0 * (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k))
                   / (SELECT COUNT(*) FROM loc), 2) AS pct_orfao;

## 99. Resumo consolidado

**Copie a saída desta célula** para o relatório ou para a base de conhecimento do agente.

In [0]:
-- 99. RESUMO CONSOLIDADO
SELECT 'VOLUMETRIA' AS bloco, 'linhas na base' AS item,
       CAST(COUNT(*) AS STRING) AS valor, '' AS veredito
  FROM base
UNION ALL
SELECT 'VOLUMETRIA', 'colunas', '23', ''
UNION ALL
SELECT 'VOLUMETRIA', 'clustering declarado',
       'cod_material, cod_centro', ''
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material + cod_centro' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material + cod_centro + cod_deposito' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material + cod_centro + cod_deposito + tp_estoque_especial' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material + cod_centro + cod_deposito + tp_estoque_especial + dateingest' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `tp_estoque_especial`, `dateingest` FROM base)
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 13**.
2. Extrair a transação no SAP com o **mesmo recorte** e na **mesma data** do snapshot.
3. Extrair **todas** as abas/telas da transação — comparar parcialmente esconde erros de granularidade.
4. Submeter os arquivos ao agente de validação junto com este notebook executado.

### Checklist antes de comparar com o SAP

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 11)
- [ ] Formato de data normalizado para `AAAAMMDD` (seção 10)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 9)
- [ ] Duplicidades classificadas: idênticas vs granularidade legítima (seção 14)
